## VARIABLES Y MÓDULOS

In [2]:
import sys, os
sys.path.append(os.path.abspath('..'))

from keras.datasets import mnist
from keras.utils import to_categorical
import numpy as np
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from keras import optimizers
from livelossplot import PlotLossesKeras
from livelossplot.outputs import MatplotlibPlot
from tensorflow.keras import callbacks
from tensorflow.keras.models import load_model
from Fuentes.imagen import DrawPanel
from PIL import Image, ImageOps


DATOS_DIR = '../Datos/'
FUENTES_DIR = '../Fuentes/'

## Ejercicio 1

La base de datos MNIST contiene imágenes de 28×28, en escala de grises, de números escritos a mano. Está conformada por 60.000 ejemplos de entrenamiento y 10.000 ejemplos de prueba.
Para cargar las imágenes utilice:

*from tensorflow.keras.datasets import mnist*

*(X_train, Y_train), (X_test, Y_test) = mnist.load_data()*

Puede visualizar una imagen utilizando:

*nImg = 0 # nro. de imagen a visualizar plt.imshow(X_train[0, :,:], cmap='gray')*

### a. Con el conjunto de 60000 imágenes entrene una red neuronal convolucional para predecir el dígito presente en la imagen. Recuerde normalizar los valores de cada imagen. Salve el modelo para recuperarlo después.

In [15]:
# Cargar y normalizar el conjunto de datos MNIST
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

Y_train= to_categorical(np.array(Y_train))
Y_test = to_categorical(np.array(Y_test))
IMG_SHAPE = X_train[0].shape
TARGET_CNT= len(Y_train[0])

print("Cantidad de imágenes de entrenamiento:", len(X_train))
print("Cantidad de imágenes de prueba:", len(X_test))
print("Forma de una imagen:", IMG_SHAPE)
print("Cantidad de clases objetivo:", TARGET_CNT)

X_train = X_train / 255
X_test  = X_test  / 255

X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test  = X_test.reshape(X_test.shape[0], 28, 28, 1)

Cantidad de imágenes de entrenamiento: 60000
Cantidad de imágenes de prueba: 10000
Forma de una imagen: (28, 28)
Cantidad de clases objetivo: 10


In [4]:
# %% Construccion del modelo
PADDING='same'
ACTIV='relu'

model = Sequential()

model.add(Input(shape=(*IMG_SHAPE, 1)))
model.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), activation=ACTIV, padding=PADDING ))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64, kernel_size=(3,3), strides=(1,1), activation=ACTIV, padding=PADDING ))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(128, kernel_size=(3,3), strides=(1,1), activation=ACTIV, padding=PADDING ))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(64, activation=ACTIV))
model.add(Dense(TARGET_CNT, activation='softmax'))

optimizer = optimizers.Adam(learning_rate=0.0001)
#optimizer = optimizers.RMSprop(learning_rate=0.0001)
#optimizer = optimizers.SGD(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'] )

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 28, 28, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 14, 14, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 7, 7, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 7, 7, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 3, 3, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1152)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │          73,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 167,114 (652.79 KB)

 Trainable params: 167,114 (652.79 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
if not os.path.exists(FUENTES_DIR+'MNIST_conv_model.keras'):
    LOTES  = 128
    EPOCAS = 100
    PACIENCIA = 10
    
    # parada temprana para evitar el sobreajuste
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True )
    visual_plot = PlotLossesKeras( outputs=[ MatplotlibPlot(figsize=(12, 5)) ] )
    
    # %% Entrenamiento del modelo usando datos de entrenamiento y validacion
    H = model.fit(x=X_train, y=Y_train, batch_size=LOTES,
                  epochs=EPOCAS,
                  validation_split=0.3,
                  callbacks=[early_stop, visual_plot],
                  verbose=0
                  )

In [9]:
model.save(FUENTES_DIR+'MNIST_conv_model.keras')

### b. Levante el modelo guardado en el punto a) y utilice la clase DrawPanel del módulo utils.images de la carpeta fuentes para generar un dibujo escrito a mano de un dígito y predecir la clase a la que pertenece.

In [3]:
model = load_model(FUENTES_DIR+'MNIST_conv_model.keras')

In [7]:
IMG_SHAPE=(28,28)

drawn_image = DrawPanel(width=200, height=200)
drawn_image.show()

image = drawn_image.get_image()
image = image.resize((28, 28)).convert('L')  # Redimensionar y convertir a escala de grises
image_array = np.array(image) / 255.0  # Normalizar
image_array = image_array.reshape(1, 28, 28, 1)  # Ajustar la forma para el modelo
prediction = model.predict(image_array)
predicted_class = np.argmax(prediction)
print("El dígito predicho es:", predicted_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
El dígito predicho es: 5


## Ejercicio 2

Se buscará resolver la clasificación de los dígitos de MNIST usando la siguiente configuración:
- model = Sequential()
- model.add(Input(shape=(28, 28, 1)))
- model.add(Conv2D(F, kernel_size=K, strides=(S,S), activation=FUN))
- model.add(MaxPooling2D(pool_size=(2,2))) # -- opcional --
- model.add(Flatten())
- model.add(Dense(10,activation='softmax'))
- model.summary()

donde F es la cantidad de filtros o de mapas de características, K es el tamaño del kernel o máscara, S es el valor del stride y FUN es la función de activación de la capa de convolución.

La tabla que aparece a continuación sugiere los valores a utilizar. Se recomienda emplear Parada Temprana para reducir el tiempo de entrenamiento. 

Para ello utilice
- from tensorflow.keras.callbacks import EarlyStopping
- es = EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001)

Esto indica que, si el valor del accuracy sobre los datos de validación no mejora después de 5 épocas, el entrenamiento finaliza. Puede usarse el parámetro min_delta para indicar cuando la diferencia entre dos accuracy se considerará significativa. Luego agregue este objeto en el momento del entrenamiento por medio del párametro callbacks

- H = model.fit(x = X_train, y = Y_train, batch_size = LOTES,
- validation_data = (X_test, Y_test), epochs=4000, callbacks=[es])

In [21]:
F = 4
K = (3,3)
S = 1
FUN = 'relu'
LOTES = 128

model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(F, kernel_size=K, strides=(S,S), activation=FUN)) 
model.add(MaxPooling2D(pool_size=(2,2))) # -- opcional -- 
model.add(Flatten())
model.add(Dense(10,activation='softmax'))
optimizer = optimizers.Adam(learning_rate=0.0001)
#optimizer = optimizers.RMSprop(learning_rate=0.0001)
#optimizer = optimizers.SGD(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'] )
model.summary()

es = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001)

H = model.fit(x = X_train, y = Y_train, batch_size = LOTES,
validation_data = (X_test, Y_test), epochs=4000, callbacks=[es])

train_acc = H.history['accuracy']

epochs = range(1, len(train_acc)+1)

print('Epocas:',epochs)
print('Accuracy en train',train_acc)

test_loss, test_acc = model.evaluate(X_test, Y_test, batch_size=128)

print(f"Accuracy en test: {test_acc:.4f}")

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)                    │ (None, 26, 26, 4)           │              40 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 13, 13, 4)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_6 (Flatten)                  │ (None, 676)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 10)                  │           6,770 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,810 (26.60 KB)

 Trainable params: 6,810 (26.60 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4934 - loss: 2.0657 - val_accuracy: 0.7963 - val_loss: 1.6240
Epoch 2/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8236 - loss: 1.1193 - val_accuracy: 0.8602 - val_loss: 0.7326
Epoch 3/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8652 - loss: 0.6044 - val_accuracy: 0.8875 - val_loss: 0.4781
Epoch 4/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8881 - loss: 0.4449 - val_accuracy: 0.9047 - val_loss: 0.3818
Epoch 5/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9009 - loss: 0.3735 - val_accuracy: 0.9144 - val_loss: 0.3320
Epoch 6/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9096 - loss: 0.3322 - val_accuracy: 0.9188 - val_loss: 0.3009
Epoch 7/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9158 - loss: 0.3044 - val_accuracy: 0.9248 - val_loss: 0.2790
Epoch 8/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9212 - loss: 0.2839 - 